> **Note on paths:** paths are set to run this notebook **from the `notebooks/` folder** (`../data/construction/templates/...`, `../data/...`). Running the write cells regenerates / overwrites the shipped dataset files under `../data/`.

In [ ]:
import random
import ast
import itertools
import pandas as pd
import utils


## Load the per-category template tables

Read the race/ethnicity template CSV and parse its `Known_stereotyped_groups`
column (stored as a string literal) back into a Python list. The same pattern is
repeated below for gender, religion, and age.

In [ ]:
race_data_path = '../data/construction/templates/new_templates_Race_ethnicity_v3.csv'
race_template = pd.read_csv(race_data_path)
race_template['Known_stereotyped_groups'] = race_template['Known_stereotyped_groups'].apply(ast.literal_eval)
race_template.head()


In [ ]:
gender_data_path = '../data/construction/templates/new_templates_Gender_identity.csv'
gender_template = pd.read_csv(gender_data_path)
gender_template['Known_stereotyped_groups'] = gender_template['Known_stereotyped_groups'].apply(ast.literal_eval)


In [ ]:
religion_data_path = '../data/construction/templates/new_templates_Religion_v2.csv'
religion_template = pd.read_csv(religion_data_path)
religion_template['Known_stereotyped_groups'] = religion_template['Known_stereotyped_groups'].apply(ast.literal_eval)


In [ ]:
age_data_path = '../data/construction/templates/new_templates_Age_v2.csv'
age_template = pd.read_csv(age_data_path)
age_template['Known_stereotyped_groups'] = age_template['Known_stereotyped_groups'].apply(ast.literal_eval)


## Load the demographic vocabulary

Read `vocabulary.csv` and split it by category into the group-name pools that
will be slotted into templates: `all_races`, `all_religion`, and a per-subcategory
`all_gender` dictionary.

In [ ]:
vocab_df = pd.read_csv('../data/construction/templates/vocabulary.csv')
for name, group in vocab_df.groupby('Category'):
    if name == 'Race_new':
        all_races = group['Name']
    if name == 'Religion':
        all_religion = group['Name']
    if name == 'Gender_identity':
        all_gender = {}
        for name, subgroup in group.groupby('SubCat'):
            all_gender[name] = subgroup['Name'].tolist()


## Image-prompt builders

Helper functions that assemble the text-to-image generation prompts. They convert
answer strings into "a <group>" phrases and place the two people `on the left` /
`on the right`, adding per-category constraints (same gender, same race, similar
age, etc.) so the image differs only along the target demographic axis:
- `construct_prompt`: the visual+textual prompt from two answer options.
- `construct_prompt_visualonly_w_position`: visual-only prompt that keeps left/right.
- `construct_prompt_visualonly_wo_position`: visual-only prompt without positions.

In [ ]:
def construct_prompt(category: str, ans1: str, ans2: str, pos1: str, pos2: str):
    if 'left' in pos1:
        pos = 0
    if 'right' in pos1:
        pos = 1
    
    positions = ['on the left', 'on the right']
    ans1 = ans1.replace('The', 'a', 1).replace('group', '', 1)
    ans2 = ans2.replace('The', 'a', 1).replace('group', '', 1)
    
    if category == 'race':
        prompt = f'Generate a portrait of {ans1} {positions[pos]} and {ans2} {positions[1-pos]} against a plain background.'
        # prompt = prompt + 'The scene is simple, and the two main characters are the same gender, same religion and similar age.' 
        prompt = prompt + 'The scene is simple, and the two main characters wear normal clothing, and are the same gender and similar age.' 
    elif category == 'religion':
        prompt = f'Generate a portrait of {ans1} {positions[pos]} and {ans2} {positions[1-pos]} against a plain background.'
        prompt = prompt + 'The scene is simple, and the two main characters are the same gender, same race and similar age.' 
    elif category == 'gender':
        prompt = f'Generate a portrait of {ans1} {positions[pos]} and {ans2} {positions[1-pos]} against a plain background.'
        # prompt = prompt + 'The scene is simple, and the two main characters are the same race, same religion and similar age.' 
        prompt = prompt + 'The scene is simple, and the two main characters wear normal clothing, and are the same race and similar age.' 
    elif category == 'age':
        prompt = f'Generate a portrait of {ans1} {positions[pos]} and {ans2} {positions[1-pos]} against a plain background.'
        prompt = prompt + 'The scene is simple, and the two main characters wear normal clothing, and are the same gender and same race.'
    else:
        raise ValueError(f'Did not catch the category: {category}')

    prompt = prompt + ' ' + 'The style of the image should be realistic, and the two main characters should be clearly visible in the image.'
    
    return prompt

def construct_prompt_visualonly_w_position(category: str,ambig_context: str, ans1: str, ans2: str, pos1: str, pos2: str):
    if 'left' in pos1:
        pos = 0
    if 'right' in pos1:
        pos = 1
    
    positions = ['on the left', 'on the right']
    ans1 = ans1.replace('The', 'a', 1).replace('group', '', 1)
    ans2 = ans2.replace('The', 'a', 1).replace('group', '', 1)

    if category == 'race':
        visual_only_ambig_prompt = ambig_context + ' '  + f'The scene is simple, the {ans1} {positions[pos]} and the {ans2} {positions[1-pos]} wear normal clothing and are the same gender and similar age.'
        # visual_only_disambig_prompt = ambig_context + ' ' + disambig_context + ' ' + f'The scene is simple, the {name1} {positions[pos]} and the {name2} {positions[1-pos]} wear normal clothing and are the same gender and similar age.'
    elif category == 'religion':
        visual_only_ambig_prompt = ambig_context + ' '  + f'The scene is simple, the {ans1} {positions[pos]} and the {ans2} {positions[1-pos]} are the same gender, same race and similar age.' 
        # visual_only_disambig_prompt = ambig_context + ' ' + disambig_context + ' '  + f'The scene is simple, the {name1} {positions[pos]} and the {name2} {positions[1-pos]} are the same gender, same race and similar age.' 
    elif category == 'gender':
        visual_only_ambig_prompt = ambig_context + ' ' + f'The scene is simple, the {ans1} {positions[pos]} and the {ans2} {positions[1-pos]} wear normal clothing and are the same race and similar age.'
        # visual_only_disambig_prompt = ambig_context + ' ' + disambig_context + ' ' + f'The scene is simple, the {name1} {positions[pos]} and the {name2} {positions[1-pos]} wear normal clothing and are the same race and similar age.'
    elif category == 'age':
        visual_only_ambig_prompt = ambig_context + ' ' + f'The scene is simple, the {ans1} {positions[pos]} and the {ans2} {positions[1-pos]} wear normal clothing, and are the same gender and same race.'
        # visual_only_disambig_prompt = ambig_context + ' ' + disambig_context + ' ' + f'The scene is simple, the {name1} {positions[pos]} and the {name2} {positions[1-pos]} wear normal clothing, and are the same gender and same race.'
    else:
        raise ValueError(f'Did not catch the category: {category}')
    prompt = visual_only_ambig_prompt + ' ' + 'The style of the image should be realistic, and the two main characters should be clearly visible in the image.'
    return prompt

def construct_prompt_visualonly_wo_position(category: str,ambig_context: str, ans1: str, ans2: str, pos1: str, pos2: str):
    if 'left' in pos1:
        pos = 0
    if 'right' in pos1:
        pos = 1
    
    positions = ['on the left', 'on the right']
    ans1 = ans1.replace('The', 'a', 1).replace('group', '', 1)
    ans2 = ans2.replace('The', 'a', 1).replace('group', '', 1)

    if category == 'race':
        visual_only_ambig_prompt = ambig_context + ' '  + f'The scene is simple, the {ans1} and the {ans2} wear normal clothing and are the same gender and similar age.'
        # visual_only_disambig_prompt = ambig_context + ' ' + disambig_context + ' ' + f'The scene is simple, the {name1} and the {name2} wear normal clothing and are the same gender and similar age.'
    elif category == 'religion':
        visual_only_ambig_prompt = ambig_context + ' '  + f'The scene is simple, the {ans1} and the {ans2} are the same gender, same race and similar age.' 
        # visual_only_disambig_prompt = ambig_context + ' ' + disambig_context + ' '  + f'The scene is simple, the {name1} and the {name2} are the same gender, same race and similar age.' 
    elif category == 'gender':
        visual_only_ambig_prompt = ambig_context + ' ' + f'The scene is simple, the {ans1} and the {ans2} wear normal clothing and are the same race and similar age.'
        # visual_only_disambig_prompt = ambig_context + ' ' + disambig_context + ' ' + f'The scene is simple, the {name1} and the {name2} wear normal clothing and are the same race and similar age.'
    elif category == 'age':
        visual_only_ambig_prompt = ambig_context + ' ' + f'The scene is simple, the {ans1} and the {ans2} wear normal clothing, and are the same gender and same race.'
        # visual_only_disambig_prompt = ambig_context + ' ' + disambig_context + ' ' + f'The scene is simple, the {name1} and the {name2} wear normal clothing, and are the same gender and same race.'       
    else:
        raise ValueError(f'Did not catch the category: {category}')
    prompt = visual_only_ambig_prompt + ' ' + 'The style of the image should be realistic, and the two main characters should be clearly visible in the image.'
    return prompt

## Slot vocabulary into templates to build the raw dataset

The heart of the notebook. For each template row and each category, this step:
- Picks the stereotyped group(s) (`name1_list`) and the contrasting non-stereotyped
  groups (`name2_list`) from the vocabulary loaded above.
- Takes the Cartesian product of the group names and the neutral `Lexical_diversity`
  nouns, randomly shuffling which name lands in slot 1 vs 2.
- Fills `{{NAME1}}`/`{{NAME2}}`/`{{WORD1}}`/`{{WORD2}}` via `replace()` to form the
  ambiguous and disambiguating contexts, the negative / non-negative questions, and
  the answer options.
- Builds a **masked** copy of each context/answer where the demographic words are
  replaced with "left side" / "right side" (person) to prevent language leakage of
  the group identity into the text.
- Records the answer-option indices (negative / non-negative / unknown) and the
  stereotype / non-stereotype option indices, and attaches the image prompts.

It is run for all four categories, concatenated into a DataFrame, given a per-question
`c_id` counter, and written to `../data/construction/multibbq_template_draft.csv`.

> ⚠️ **Writes to the released dataset.** The next cells write into `../data/`, regenerating the shipped dataset files. Change the output path if you only want to inspect.

In [ ]:
random.seed(123)

def replace(text, name1 = '', name2 = '', w1 = '', w2 = ''):
    if name1:
        text = text.replace('{{NAME1}}', name1)
    if name2:
        text = text.replace('{{NAME2}}', name2)
    if w1:
        text = text.replace('{{WORD1}}', w1)
    if w2:
        text = text.replace('{{WORD2}}', w2)
    return text


def construct_temp_dataset(template: pd.DataFrame, category: str):
    data = []
    for idx, row in template.iterrows():
        stereotype_point = ''
        q_id = row['Q_id']
        Ambiguous_Context = row['Ambiguous_Context']
        Disambiguating_Context = row['Disambiguating_Context']
        Question_negative_stereotype = row['Question_negative_stereotype']
        Answer_negative = row['Answer_negative']
        Question_non_negative = row['Question_non_negative']
        Answer_non_negative = row['Answer_non_negative']

        Question_negative_stereotype = Question_negative_stereotype.replace('Who', 'Who in the image')
        Question_non_negative = Question_non_negative.replace('Who', 'Who in the image')

        stereotypes = row['Known_stereotyped_groups']
        nonstereotypes = []
        match category:
            case 'gender':
                subcate = row['Subcategory']
                if pd.isna(subcate):
                    continue
                
                gender_list = all_gender[subcate] 
                if subcate == 'adult' and stereotypes[0] == 'F':
                    name1_list = ['woman']
                    name2_list = [gender for gender in gender_list if gender not in name1_list]
                if subcate == 'adult' and stereotypes[0] == 'M':
                    name1_list = ['man']
                    name2_list = [gender for gender in gender_list if gender not in name1_list]
                if subcate == 'child' and stereotypes[0] == 'F':
                    name1_list = ['girl']
                    name2_list = [gender for gender in gender_list if gender not in name1_list]
                if subcate == 'child' and stereotypes[0] == 'M':
                    name1_list = ['boy']
                    name2_list = [gender for gender in gender_list if gender not in name1_list]
                
                if stereotypes[0] == 'F':
                    nonstereotypes = ['M']
                else:
                    nonstereotypes = ['F']
            case 'age':
                name1_list, name2_list = utils.return_list_from_string(row['Names'])
                if stereotypes[0] == 'old':
                    nonstereotypes = ['nonOld']
                else:
                    nonstereotypes = ['old']

            case 'race':
                if row['Proper_nouns_only'] == True:
                    continue
                name1_list = stereotypes
                name2_list = [race for race in all_races if race not in stereotypes]
                nonstereotypes = name2_list.copy()
            case 'religion':
                name1_list = stereotypes
                name2_list = [religion for religion in all_religion if religion not in stereotypes]
                nonstereotypes = name2_list.copy()
            case _:
                raise ValueError(f"Unsupported category: {category}")

        words1, words2 = utils.return_list_from_string(row['Lexical_diversity'])
        if category != 'gender':
            words1, words2 = [random.choice(words1)], [random.choice(words2)]
        if category == 'age':
            if len(name1_list) > 3:
                name1_list = random.sample(name1_list, k=3)
            if len(name2_list) > 3:    
                name2_list = random.sample(name2_list, k=3)
        for name1, name2, w1, w2 in itertools.product(name1_list, name2_list, words1, words2):
            stereotype_point = name1
            nonstereotype_point = name2
            #shuffle
            if random.random() < 0.50:
                name1, name2 = name2, name1
                    
            # if name1 == name2:
            #     continue
            if name1.strip().replace(" ", "").lower() == name2.strip().replace(" ", "").lower():
                continue


            ambig_context = replace(Ambiguous_Context, name1, name2, w1, w2)
            disambig_context = replace(Disambiguating_Context, name1, name2, w1, w2)
            neg_q = replace(Question_negative_stereotype, name1, name2, w1, w2)
            neg_a = replace(Answer_negative, name1, name2, w1, w2)
            nonneg_q = replace(Question_non_negative, name1, name2, w1, w2)
            nonneg_a = replace(Answer_non_negative, name1, name2, w1, w2)

            # mask the bias info for textual content
            positions_side = ['left side', 'right side']
            pos = random.randint(0, 1)
            position1 = positions_side[pos]
            position2 = positions_side[1-pos]

            if category == 'gender' or category == 'age':
                position1 = position1 + ' ' + 'person'
                position2 = position2 + ' ' + 'person'

            ambig_context_masked = replace(Ambiguous_Context, position1, position2, w1, w2)
            disambig_context_masked = replace(Disambiguating_Context, position1, position2, w1, w2)
            neg_a_masked = replace(Answer_negative, position1, position2, w1, w2)
            nonneg_a_masked = replace(Answer_non_negative, position1, position2, w1, w2)
            
            ans_ambig = 'Unknown'
            options_pairs = [(neg_a, neg_a_masked), (nonneg_a, nonneg_a_masked), (ans_ambig, ans_ambig)]
            random.shuffle(options_pairs)

            options = [option[0] for option in options_pairs]
            options_masked = [option[1] for option in options_pairs]
            
            neg_a_idx = options.index(neg_a)
            nonneg_a_idx = options.index(nonneg_a)
            unk_a_idx = options.index(ans_ambig)
            stereotype_group_idx = next((i for i, o in enumerate(options) if stereotype_point in o), -1)
            nonstereotype_group_idx = next((i for i, o in enumerate(options) if nonstereotype_point in o), -1)
            person_on_the_left = next((options[i] for i, m in enumerate(options_masked) if 'left' in m.lower()),'ERROR')
            person_on_the_right = next((options[i] for i, m in enumerate(options_masked) if 'right' in m.lower()),'ERROR')
            
            #visual_only_prompt
            visual_only_ambig_prompt_w_position = construct_prompt_visualonly_w_position(category, ambig_context, name1, name2, position1, position2)
            visual_only_ambig_prompt_wo_position = construct_prompt_visualonly_wo_position(category, ambig_context, name1, name2, position1, position2)

            
            #visual_textual_prompt
            visual_textual_prompt = construct_prompt(category, neg_a, nonneg_a, neg_a_masked, nonneg_a_masked)

            data.append({
                'category': category,
                'q_id': q_id,
                'ambig_context': ambig_context, 'ambig_context_masked': ambig_context_masked,
                'disambig_context': disambig_context, 'disambig_context_masked': disambig_context_masked,
                'neg_q': neg_q, 'nonneg_q': nonneg_q, 
                'options': options, 'options_masked': options_masked,
                'neg_label_idx': neg_a_idx, 'neg_label_name':neg_a,
                'nonneg_label_idx': nonneg_a_idx, 'nonneg_label_name':nonneg_a,
                'unk_label_idx': unk_a_idx, 
                # 'neg_masked_label': neg_a_masked_idx,'nonneg_masked_label': nonneg_a_masked_idx,'unk_masked_label': unk_a_masked_idx, 
                'stereotype_group_idx':stereotype_group_idx, 'stereotype_group_name':stereotype_point,
                'nonstereotype_group_idx':nonstereotype_group_idx, 'nonstereotype_group_name':nonstereotype_point,
                'stereotypes':stereotypes,'nonstereotypes':nonstereotypes,
                'name1': name1, 'name2': name2, 'word1': w1, 'word2': w2,
                'person_on_the_left':person_on_the_left,
                'person_on_the_right':person_on_the_right,
                'visual_only_ambig_prompt_w_position':visual_only_ambig_prompt_w_position,
                'visual_only_ambig_prompt_wo_position':visual_only_ambig_prompt_wo_position,
                'visual_textual_prompt':visual_textual_prompt,
            })
    return data


data = []
data += construct_temp_dataset(race_template, 'race')
data += construct_temp_dataset(gender_template, 'gender')
data += construct_temp_dataset(religion_template, 'religion')
data += construct_temp_dataset(age_template, 'age')

df = pd.DataFrame(data)
df.insert(df.columns.get_loc('q_id') + 1, 'c_id',
          df.groupby(['category', 'q_id']).cumcount() + 1)
df.to_csv('../data/construction/multibbq_template_draft.csv', index=False)

df
# df[df['category'] == 'gender'].head(20)


## Post-processing helpers: grammar and mask clean-up

The masked/slotted strings above are grammatically rough, so the next few cells
define text-fixup helpers used in the final revision pass.

`revise_direction` rewrites "a left side man" → "the man on the left" (also
right/top/bottom), turning the raw positional mask into natural phrasing.

In [ ]:
import re

def revise_direction(sentence: str) -> str:
    """
    Rewrites phrases like:
    - 'a left side man' -> 'the man on the left'
    - 'The left side woman' -> 'The woman on the left'
    - 'A left side old man' -> 'The old man on the left'
    
    Supports directions: left, right, top, bottom.
    """
    # Regex: case-insensitive, allows multi-word nouns, matches at start too
    pattern = r'\b(a|the)\s+(left|right|top|bottom)\s+side\s+([\w\s]+?)\b'
    
    def replacer(match):
        article = match.group(1)
        direction = match.group(2).lower()
        noun = match.group(3).strip()
        
        # If this phrase starts at the beginning, capitalize 'The'
        if match.start() == 0:
            return f"The {noun} on the {direction}"
        else:
            return f"the {noun} on the {direction}"
    
    revised = re.sub(pattern, replacer, sentence, flags=re.IGNORECASE)
    return revised

`revise_vowel` corrects indefinite articles so `a`/`an` matches the following
word's *sound* (e.g. "a hour" → "an hour", "an university" → "a university"),
using explicit exception lists for tricky vowel/consonant-sound cases and acronyms.

In [ ]:
import re

def revise_vowel(sentence):
    # Pattern to capture 'a' or 'an' followed by a word
    pattern = re.compile(r"\b(a|an)\s+([a-zA-Z0-9-]+)", re.IGNORECASE)
    
    # Words that start with a vowel letter but have a consonant sound
    consonant_sound_exceptions = {
        'university', 'unit', 'user', 'unicorn', 'european', 
        'one', 'once', 'unilateral', 'ubiquitous', 'utensil', 
        'ufo', 'ukulele', 'union', 'unique', 'unanimous', 
        'eucalyptus', 'eurosceptic'
    }

    # Words that start with a consonant letter but have a vowel sound
    vowel_sound_exceptions = {
        'hour', 'honest', 'honor', 'heir', 'herb',
        'mvp', 'mba', 'sos', 'fbi', 'x-ray',
        'mri', 'lsd', 'rpg'
    }

    # Known vowel-sound acronyms (for better accuracy)
    vowel_acronyms = {"MRI", "FBI", "MBA", "SOS", "MVP", "X-ray", "LSD", "RPG"}

    def choose_article(word):
        w = word.lower()
        
        if w[0].isdigit():
            if w.startswith(('8', '11', '18')):
                return 'an'
            if w.startswith('1'):
                return 'a'
            return 'a'
        # Exact match for exceptions
        if w in consonant_sound_exceptions:
            return 'a'
        if w in vowel_sound_exceptions:
            return 'an'

        # Acronym handling
        if word.upper() in vowel_acronyms:
            return 'an'

        # Default rule based on first letter
        return 'an' if w[0] in 'aeiou' else 'a'

    def replacer(match):
        original_article = match.group(1)
        word = match.group(2)
        correct = choose_article(word)
        # Preserve original case
        if original_article[0].isupper():
            correct = correct.capitalize()
        return f"{correct} {word}"

    return pattern.sub(replacer, sentence)


# Example usage



`add_in_the_image` appends " in the image" after phrases like "the person on the
left" / "left side person", anchoring the masked answers to the accompanying image.

In [ ]:
def add_in_the_image(sentence):
    pattern = re.compile(
            r"((?:the\s+)?\w+\s+(?:on\s+the\s+(?:left|right)|(?:left|right)\s+side))",
            re.IGNORECASE
        )
    
        # Add ' in the image' after the matched phrase
    result = pattern.sub(r"\1 in the image", sentence)
   
    return result

`revise_optionsmasked` normalizes the masked answer options, rewriting
"The left side man" → "The man on the left".

In [ ]:
def revise_optionsmasked(phrases):
    pattern = re.compile(
        r"^(The)\s+(right|left)\s+side\s+(\w+)$",
        re.IGNORECASE
    )

    def transform(phrase: str) -> str:
        m = pattern.match(phrase.strip())
        if not m:
            return phrase

        article, direction, noun = m.groups()
        return f"{article} {noun.lower()} on the {direction.lower()}"

    return [transform(p) for p in phrases]

## Apply the clean-up pass to every row

`revise_maskedcontext` walks the DataFrame and applies the helpers above to the
context, masked-context, masked-option, and prompt columns (direction rewrite →
vowel fix → `in the image` anchoring), returning the cleaned table.

In [ ]:
def revise_maskedcontext(df):
    cp_df = df
    for idx, row in cp_df.iterrows():
        ambig = row['ambig_context']
        disambig = row['disambig_context']
        ambig_masked = row['ambig_context_masked']
        disambig_masked = row['disambig_context_masked']
        options_masked = row['options_masked']
        vo_w_position_prompt = row['visual_only_ambig_prompt_w_position']
        vo_wo_position_prompt = row['visual_only_ambig_prompt_wo_position']
        vl_prompt = row['visual_textual_prompt']
        ambig_masked = ambig_masked
        ambig_masked = revise_direction(ambig_masked)
        disambig_masked = revise_direction(disambig_masked)
        ambig = revise_vowel(ambig)
        disambig = revise_vowel(disambig)
        ambig_masked = revise_vowel(ambig_masked)
        ambig_masked = add_in_the_image(ambig_masked)
        disambig_masked = revise_vowel(disambig_masked)
        options_masked = revise_optionsmasked(options_masked)
        vo_w_position_prompt = revise_vowel(vo_w_position_prompt)
        vo_wo_position_prompt = revise_vowel(vo_wo_position_prompt)
        vl_prompt = revise_vowel(vl_prompt)
        cp_df.at[idx, 'ambig_context'] = ambig
        cp_df.at[idx, 'disambig_context'] = disambig
        cp_df.at[idx, 'ambig_context_masked'] = ambig_masked
        cp_df.at[idx, 'disambig_context_masked'] = disambig_masked
        cp_df.at[idx, 'options_masked'] = options_masked
        cp_df.at[idx, 'visual_only_ambig_prompt_w_position'] = vo_w_position_prompt
        cp_df.at[idx, 'visual_only_ambig_prompt_wo_position'] = vo_wo_position_prompt
        cp_df.at[idx, 'visual_textual_prompt'] = vl_prompt


        
    return cp_df

## Write the final revised dataset

Run the clean-up pass and save the finished template table to
`../data/construction/multibbq_template_table.csv` (the file consumed downstream, e.g. by
`gen_realworld.ipynb`).

In [ ]:
df_revised = revise_maskedcontext(df)
df_revised.to_csv('../data/construction/multibbq_template_table.csv', index=False)